In [1]:
import os, gc, random, warnings
import numpy as np
import pandas as pd
import librosa
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import timm
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
from torch.cuda.amp import autocast, GradScaler

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
print("HF Auth OK")

class CFG:
    base_dir = Path('/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup')
    stems_dir = base_dir / 'genres_stems'
    noise_dir = base_dir / 'ESC-50-master' / 'audio'
    mashups_dir = base_dir / 'mashups'
    test_csv = base_dir / 'test.csv'
    out_dir = Path('/kaggle/working')
    out_dir.mkdir(exist_ok=True)
    
    sr = 16000
    dur = 10
    n_samples = sr * dur
    full_dur = 30
    full_samples = sr * full_dur
    
    model_name = 'resnet50'
    epochs = 25
    bs = 128  # 64 per T4
    lr = 2e-4 # Stabilized LR
    
    mixup_p = 0.4
    mixup_alpha = 0.4
    tta_crops = 9
    
    genres = ['blues', 'classical', 'country', 'disco', 'hiphop',
              'jazz', 'metal', 'pop', 'reggae', 'rock']
    g2i = {g: i for i, g in enumerate(genres)}
    i2g = {i: g for i, g in enumerate(genres)}

class AudioCache:
    def __init__(self):
        self.stems = {g: [] for g in CFG.genres}
        self.noises = []
        
        print("Caching 16kHz Audio into RAM...")
        for g in tqdm(CFG.genres, desc="Stems"):
            g_path = CFG.stems_dir / g
            for song_dir in g_path.iterdir():
                if song_dir.is_dir():
                    song_stems = {}
                    for st in ['bass.wav', 'drums.wav', 'other.wav', 'vocals.wav']:
                        p = song_dir / st
                        if p.exists():
                            y, _ = librosa.load(p, sr=CFG.sr, mono=True)
                            song_stems[st] = y.astype(np.float32)
                    if len(song_stems) == 4:
                        self.stems[g].append(song_stems)
                        
        for p in CFG.noise_dir.glob('*.wav'):
            y, _ = librosa.load(p, sr=CFG.sr, duration=5, mono=True)
            if len(y) > 0:
                self.noises.append(y.astype(np.float32))

cache = AudioCache()
gc.collect()

class MashupDataset(Dataset):
    def __init__(self, cache, n_samples, is_train=True):
        self.cache = cache
        self.n = n_samples
        self.is_train = is_train

    def __len__(self):
        return self.n

    def __getitem__(self, idx):
        g = random.choice(CFG.genres)
        label = CFG.g2i[g]
        
        songs = random.sample(self.cache.stems[g], 4)
        stems = [songs[0]['bass.wav'], songs[1]['drums.wav'], 
                 songs[2]['other.wav'], songs[3]['vocals.wav']]
        
        mix = np.zeros(CFG.full_samples, dtype=np.float32)
        for s in stems:
            if len(s) < CFG.full_samples:
                s = np.pad(s, (0, CFG.full_samples - len(s)))
            else:
                s = s[:CFG.full_samples]
            
            vol = random.uniform(0.8, 1.2) if self.is_train else 1.0
            mix += s * vol

        rms = np.sqrt(np.mean(mix**2)) + 1e-8
        mix = mix * (10**(-20/20) / rms)
        
        if self.is_train:
            noise = random.choice(self.cache.noises)
            snr = random.uniform(10, 25)
            
            if len(noise) < CFG.full_samples:
                reps = int(np.ceil(CFG.full_samples / len(noise)))
                noise = np.tile(noise, reps)[:CFG.full_samples]
            else:
                st = random.randint(0, len(noise) - CFG.full_samples)
                noise = noise[st:st+CFG.full_samples]
                
            c_pow, n_pow = np.mean(mix**2) + 1e-10, np.mean(noise**2) + 1e-10
            scale = np.sqrt(c_pow / (n_pow * 10**(snr/10)))
            mix += noise * scale

        mix = np.clip(mix, -1.0, 1.0)
        
        st = random.randint(0, CFG.full_samples - CFG.n_samples)
        crop = mix[st:st+CFG.n_samples]
        
        return torch.tensor(crop, dtype=torch.float32), label

class TorchaudioFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=CFG.sr, n_fft=2048, hop_length=512, f_min=20, f_max=8000, n_mels=128
        )
        self.a2db = torchaudio.transforms.AmplitudeToDB()
        self.freq_mask = torchaudio.transforms.FrequencyMasking(24)
        self.time_mask = torchaudio.transforms.TimeMasking(48)

    def forward(self, x):
        # Force FP32 to prevent Autocast NaNs
        with torch.cuda.amp.autocast(enabled=False):
            x = x.float()
            x = self.mel(x)
            x = self.a2db(x)
            
            m = x.mean(dim=[-2, -1], keepdim=True)
            s = x.std(dim=[-2, -1], keepdim=True)
            x = (x - m) / (s + 1e-5)
            
            if self.training:
                if random.random() < 0.5: x = self.freq_mask(x)
                if random.random() < 0.5: x = self.time_mask(x)
                
            x = x.unsqueeze(1) 
        return x

def train():
    print(f"\n--- Training {CFG.model_name} ---")
    
    train_dl = DataLoader(MashupDataset(cache, 8000, True), batch_size=CFG.bs, shuffle=True, num_workers=4, pin_memory=True)
    val_dl = DataLoader(MashupDataset(cache, 1500, False), batch_size=CFG.bs, shuffle=False, num_workers=4, pin_memory=True)
    
    fe = TorchaudioFeatureExtractor().to(device)
    model = timm.create_model(CFG.model_name, pretrained=True, in_chans=1, num_classes=10).to(device)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=CFG.lr, total_steps=len(train_dl)*CFG.epochs, pct_start=0.1)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    scaler = GradScaler()
    
    best_f1 = 0
    
    for ep in range(CFG.epochs):
        model.train()
        fe.train()
        
        for x, y in train_dl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            
            # Extract features in FP32
            x_feats = fe(x)
            
            # Mixup on 2D Spectrogram features
            if random.random() < CFG.mixup_p:
                lam = np.random.beta(CFG.mixup_alpha, CFG.mixup_alpha)
                idx = torch.randperm(x_feats.size(0)).to(device)
                x_feats = lam * x_feats + (1 - lam) * x_feats[idx]
                y_b = y[idx]
            else:
                lam = 1.0
                y_b = y
            
            with autocast():
                logits = model(x_feats)
                if lam == 1.0:
                    loss = criterion(logits, y)
                else:
                    loss = lam * criterion(logits, y) + (1 - lam) * criterion(logits, y_b)
                    
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sched.step()
            
        model.eval()
        fe.eval()
        preds, tgts = [], []
        
        with torch.no_grad():
            for x, y in val_dl:
                x = x.to(device)
                x_feats = fe(x)
                with autocast():
                    logits = model(x_feats)
                preds.extend(logits.argmax(1).cpu().numpy())
                tgts.extend(y.numpy())
                
        val_f1 = f1_score(tgts, preds, average='macro')
        print(f"Ep {ep+1}/{CFG.epochs} | Val F1: {val_f1:.4f}")
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            torch.save(state, CFG.out_dir / f'{CFG.model_name}_best.pth')

    print(f"Best F1: {best_f1:.4f}")

train()

del cache
gc.collect()

# Inference
print("\n--- Running TTA Inference ---")
test_df = pd.read_csv(CFG.test_csv)
all_preds = np.zeros((len(test_df), 10))

def get_crops(path):
    try:
        y, _ = librosa.load(path, sr=CFG.sr, duration=30, mono=True)
    except:
        y = np.zeros(CFG.full_samples, dtype=np.float32)
        
    if len(y) < CFG.full_samples:
        y = np.pad(y, (0, CFG.full_samples - len(y)))
    else:
        y = y[:CFG.full_samples]
        
    rms = np.sqrt(np.mean(y**2)) + 1e-8
    y = np.clip(y * (10**(-20/20) / rms), -1.0, 1.0)
    
    max_st = CFG.full_samples - CFG.n_samples
    starts = np.linspace(0, max_st, CFG.tta_crops, dtype=int)
    return torch.tensor(np.array([y[s:s+CFG.n_samples] for s in starts]), dtype=torch.float32)

fe = TorchaudioFeatureExtractor().to(device)
fe.eval()

model = timm.create_model(CFG.model_name, pretrained=False, in_chans=1, num_classes=10).to(device)
model.load_state_dict(torch.load(CFG.out_dir / f'{CFG.model_name}_best.pth'))
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)
model.eval()

with torch.no_grad():
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
        crops = get_crops(CFG.mashups_dir / f"song{str(row['id']).zfill(4)}.wav").to(device)
        x_feats = fe(crops)
        with autocast():
            logits = model(x_feats)
        probs = F.softmax(logits, dim=1).mean(dim=0).cpu().numpy()
        all_preds[idx] = probs

test_df['genre'] = [CFG.i2g[i] for i in all_preds.argmax(axis=1)]
test_df['id'] = test_df['id'].astype(str).str.zfill(4)
test_df.to_csv(CFG.out_dir / 'submission.csv', index=False)
print("Inference Complete! Saved to submission.csv")

HF Auth OK
Caching 16kHz Audio into RAM...


Stems:   0%|          | 0/10 [00:00<?, ?it/s]


--- Training resnet50 ---


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

Ep 1/25 | Val F1: 0.1433
Ep 2/25 | Val F1: 0.3137
Ep 3/25 | Val F1: 0.6558
Ep 4/25 | Val F1: 0.7648
Ep 5/25 | Val F1: 0.8072
Ep 6/25 | Val F1: 0.8267
Ep 7/25 | Val F1: 0.8545
Ep 8/25 | Val F1: 0.8874
Ep 9/25 | Val F1: 0.9020
Ep 10/25 | Val F1: 0.9014
Ep 11/25 | Val F1: 0.9294
Ep 12/25 | Val F1: 0.9198
Ep 13/25 | Val F1: 0.9296
Ep 14/25 | Val F1: 0.9285
Ep 15/25 | Val F1: 0.9277
Ep 16/25 | Val F1: 0.9294
Ep 17/25 | Val F1: 0.9491
Ep 18/25 | Val F1: 0.9315
Ep 19/25 | Val F1: 0.9271
Ep 20/25 | Val F1: 0.9447
Ep 21/25 | Val F1: 0.9509
Ep 22/25 | Val F1: 0.9444
Ep 23/25 | Val F1: 0.9430
Ep 24/25 | Val F1: 0.9366
Ep 25/25 | Val F1: 0.9374
Best F1: 0.9509

--- Running TTA Inference ---


Inference:   0%|          | 0/3020 [00:00<?, ?it/s]

Inference Complete! Saved to submission.csv
